# <div align="center">🌟 Madar (مدار) - Cloud LoRA Fine-Tuning Studio</div>
### <div align="center">Train LLMs natively in Ring/Madar with RingTensor & GPU Acceleration</div>

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azzeddine2017/Madar/blob/main/notebooks/Madar_LoRA_Studio.ipynb) 
[![GitHub](https://img.shields.io/badge/GitHub-Madar-blue.svg)](https://github.com/azzeddine2017/Madar) 
[![RingTensor](https://img.shields.io/badge/Autograd-RingTensor%20C%20Kernels-green.svg)]() 
[![LoRA](https://img.shields.io/badge/Fine--Tuning-Low--Rank%20Adaptation-purple.svg)]()

---

Welcome to the **Madar LoRA Fine-Tuning Studio** on Google Colab.

This studio trains and fine-tunes LLMs (Qwen2.5 / LLaMA / BitNet) **100% natively in the Madar dialect** using the **Madar LoRA Engine** (`libraries/madar_ai/trainer/`) accelerated by **RingTensor C Autograd and OpenCL GPU kernels**.

### 🎯 Key Highlights:
- **Zero Python ML Overhead:** The forward pass, token masking, backward pass, and AdamW optimizer run directly in the Madar VM and C extensions.
- **Curated Ring Documentation Dataset:** Ingests **730 high-density instruction-response pairs** synthesized directly from the official Ring language Sphinx documentation.
- **Hardware Acceleration:** Native AVX2/FMA vectorization on CPU + OpenCL Zero-Copy GEMM on NVIDIA Tesla GPUs (T4 / V100 / A100).
- **Portable Checkpoints:** Exports lightweight standalone LoRA adapter weights (`.tensor` and `.json`) ready for deployment or local execution.

## ⚙️ Step 1: Hardware & Environment Verification
Verify the attached GPU accelerator (e.g. Tesla T4) and CPU SIMD features.

In [ ]:
# 1. Check NVIDIA GPU status
!nvidia-smi

# 2. Check CPU AVX2 & FMA instruction support
!lscpu | grep -E "Model name|avx2|fma"

## 🔨 Step 2: Clone Repository, Build Runtime & Extensions
Installs dependencies, builds the Core VM, and compiles `libring_tensor.so` (with OpenCL) and `libring_cjson.so`.

In [ ]:
import os, sys

# 1. Clone repository if not present or pull latest updates
if not os.path.exists("/content/Madar/language"):
    !git clone https://github.com/azzeddine2017/Madar.git /content/Madar
    %cd /content/Madar
else:
    %cd /content/Madar
    !git pull

# 2. Install essential packages for OpenCL, OpenMP, and GCC
!apt-get update -qq && apt-get install -y -qq build-essential g++ ocl-icd-opencl-dev opencl-headers curl libcurl4-openssl-dev

# 3. Execute the Linux build pipeline for Core Runtime
!bash tools/build_linux.sh

# 4. Apply clean macro renames to prevent C symbol collisions with core ring_tensor.h
header_path = "/content/Madar/extensions/ringtensor/ring_tensor.h"
if os.path.exists(header_path):
    with open(header_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = content.replace("#define ring_tensor_h\n", "").replace("#define ring_tensor_h", "")
    if "rt_ext_api_reshape" not in content:
        renames = """
#define ring_tensor_new            rt_ext_api_new
#define ring_tensor_new_from_buffer rt_ext_api_new_from_buffer
#define ring_tensor_delete         rt_ext_api_delete
#define ring_tensor_create_view    rt_ext_api_create_view
#define ring_tensor_slice          rt_ext_api_slice
#define ring_tensor_reshape        rt_ext_api_reshape
#define ring_tensor_print          rt_ext_api_print
#define ring_tensor_to_list        rt_ext_api_to_list
#define ring_tensor_fill           rt_ext_api_fill
#define ring_tensor_add            rt_ext_api_add
#define ring_tensor_sub            rt_ext_api_sub
#define ring_tensor_mul            rt_ext_api_mul
#define ring_tensor_div            rt_ext_api_div
#define ring_tensor_scale          rt_ext_api_scale
#define ring_tensor_add_scalar     rt_ext_api_add_scalar
#define ring_tensor_matmul         rt_ext_api_matmul
#define ring_tensor_relu           rt_ext_api_relu
#define ring_tensor_gelu           rt_ext_api_gelu
#define ring_tensor_sigmoid        rt_ext_api_sigmoid
#define ring_tensor_softmax        rt_ext_api_softmax
#define ring_tensor_sum            rt_ext_api_sum
#define ring_tensor_mean           rt_ext_api_mean
#define ring_tensor_max            rt_ext_api_max
#define ring_tensor_min            rt_ext_api_min
#define ring_tensor_cosine_similarity rt_ext_api_cosine_similarity
#define ring_tensor_element_size   rt_ext_api_element_size
#define ring_tensor_total_elements rt_ext_api_total_elements
#define ring_tensor_data           rt_ext_api_data
#define ring_tensor_loadfunctions  rt_ext_api_loadfunctions
"""
        content = content.replace('#include "ring.h"', '#include "ring.h"\n' + renames)
        with open(header_path, "w", encoding="utf-8") as f:
            f.write(content)

# Fix USE_OPENCL in ring_tensor.c
c_path = "/content/Madar/extensions/ringtensor/ring_tensor.c"
if os.path.exists(c_path):
    with open(c_path, "r", encoding="utf-8") as f:
        c_code = f.read()
    if "#ifndef USE_OPENCL" not in c_code:
        c_code = c_code.replace("#define USE_OPENCL 1", "#ifndef USE_OPENCL\n#define USE_OPENCL 1\n#endif")
        with open(c_path, "w", encoding="utf-8") as f:
            f.write(c_code)

# 5. Direct Compilation & Installation of RingTensor Shared Library
!mkdir -p /content/Madar/lib /content/Madar/bin
!gcc -shared -o /content/Madar/lib/libring_tensor.so -O3 -fPIC -fopenmp \
    /content/Madar/extensions/ringtensor/ring_tensor.c \
    -I /content/Madar/extensions/ringtensor/include \
    -I /content/Madar/language/include \
    -L /content/Madar/lib -Wl,-rpath,'$ORIGIN':-Wl,-rpath,'$ORIGIN/../lib' -lring -lOpenCL 2>/dev/null || \
 gcc -shared -o /content/Madar/lib/libring_tensor.so -O3 -fPIC -fopenmp -DUSE_OPENCL=0 \
    /content/Madar/extensions/ringtensor/ring_tensor.c \
    -I /content/Madar/extensions/ringtensor/include \
    -I /content/Madar/language/include \
    -L /content/Madar/lib -Wl,-rpath,'$ORIGIN':-Wl,-rpath,'$ORIGIN/../lib' -lring

!cp -f /content/Madar/lib/libring_tensor.so /content/Madar/bin/libring_tensor.so
!cp -f /content/Madar/lib/libring_tensor.so /content/Madar/extensions/ringtensor/libring_tensor.so

# 6. Direct Compilation of RingCJSON (Required for Dataset Parsing)
!if [ -d "/content/Madar/extensions/ringcjson" ]; then \
    cd /content/Madar/extensions/ringcjson && \
    gcc -c -fpic -O2 ring_cjson.c lib/cJSON.c lib/cJSON_Utils.c -I /content/Madar/language/include && \
    gcc -shared -o /content/Madar/lib/libring_cjson.so ring_cjson.o cJSON.o cJSON_Utils.o -L /content/Madar/lib -Wl,-rpath,'$ORIGIN':-Wl,-rpath,'$ORIGIN/../lib' -lring && \
    cp -f /content/Madar/lib/libring_cjson.so /content/Madar/bin/libring_cjson.so && \
    cp -f /content/Madar/lib/libring_cjson.so /content/Madar/extensions/ringcjson/libring_cjson.so && \
    rm -f ring_cjson.o cJSON.o cJSON_Utils.o && \
    echo "  [OK] RingCJSON compiled successfully."; \
fi

# 7. Set runtime library paths
os.environ["LD_LIBRARY_PATH"] = "/content/Madar/lib:/content/Madar/bin:" + os.environ.get("LD_LIBRARY_PATH", "")

# 8. Verify library presence
!ls -lh /content/Madar/lib/libring_tensor.so
!ls -lh /content/Madar/lib/libring_cjson.so
!bin/ring -v

## 📥 Step 3: Download Pre-Trained Base Model
Download the high-speed Qwen2.5-0.5B-Instruct GGUF base model for fine-tuning.

In [ ]:
!mkdir -p models
# Download base Qwen2.5 0.5B Instruct GGUF
!wget -c -q --show-progress -O models/qwen2.5-0.5b-instruct-q4_0.gguf \
    https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_0.gguf

!ls -lh models/

## 📚 Step 4: Verify Instruction Training Dataset
Verify the synthesized 730 Ring documentation training pairs in `data/ring_instruct_dataset.jsonl`.

In [ ]:
# Verify dataset line count and file size
!wc -l data/ring_instruct_dataset.jsonl
!head -n 2 data/ring_instruct_dataset.jsonl

## 🪄 Step 5: Register Interactive `%%madar` Cell Magic
Allows executing Madar / Ring code directly within notebook cells.

In [ ]:
from IPython.core.magic import register_cell_magic
import subprocess, tempfile, os, sys

MADAR_BIN = os.path.abspath("bin/ring")

@register_cell_magic
def madar(line, cell):
    """IPython Cell Magic to execute Madar code directly."""
    with tempfile.NamedTemporaryFile("w", suffix=".ring", delete=False, encoding="utf-8") as f:
        f.write(cell)
        tmp_file = f.name
    try:
        env = os.environ.copy()
        env["LD_LIBRARY_PATH"] = "/content/Madar/lib:/content/Madar/bin:/content/Madar/extensions/ringtensor:" + env.get("LD_LIBRARY_PATH", "")
        cwd = os.path.abspath("/content/Madar") if os.path.exists("/content/Madar") else os.path.abspath(".")
        res = subprocess.run([MADAR_BIN, tmp_file], capture_output=True, text=True, env=env, cwd=cwd)
        if res.stdout:
            print(res.stdout, end="")
        if res.stderr:
            print(res.stderr, end="", file=sys.stderr)
    finally:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

ip = get_ipython()
ip.register_magic_function(madar, 'cell', 'ring')
print("✅ %%madar cell magic registered successfully!")

## 🔍 Step 6: Baseline Evaluation (Before Fine-Tuning)
Evaluate base model responses to Ring programming queries before applying LoRA.

In [ ]:
%%madar
load "madar_ai.ring"

? "=== Baseline Evaluation: Pre-Trained Model ==="
oModel = model_open("models/qwen2.5-0.5b-instruct-q4_0.gguf")
oModel.setTemperature(0.2)
oModel.setMaxTokens(128)

cQ = "Write a Ring code example to reverse a string and check if it is a palindrome."
cPrompt = oModel.formatChatPrompt(cQ, "You are an expert Ring language programmer.")
? oModel.generate(cPrompt, 120)


## 🚀 Step 7: Launch Full Madar LoRA Fine-Tuning
Train the LoRA adapters on all **730 instruction pairs** using **RingTensor C Autograd & AdamW** with Tesla T4 GPU acceleration.

In [ ]:
%%madar
load "madar_ai.ring"
load "madar_trainer.ring"

cModelPath = "models/qwen2.5-0.5b-instruct-q4_0.gguf"
cDataPath  = "data/ring_instruct_dataset.jsonl"

? "===================================================================="
? "  Madar AI - Full Cloud LoRA Training Execution (Colab GPU)           "
? "===================================================================="

# 1. Initialize Base Model with OpenCL GPU Acceleration
oModel = model_open(cModelPath)
oModel.setThreads(4)
oModel.setGPU(true)

# 2. Load the 730 Ring Instruction Pairs
oDataset = new MadarInstructDataset(cDataPath, "chatml")
? f"Loaded Dataset: {oDataset.size()} Instruction-Response Pairs"

# 3. Configure LoRA Trainer:
# Rank: 16 | Alpha: 32 | Base LR: 0.0002
oTrainer = new MadarLoRATrainer(oModel, oDataset, 16, 32.0, 0.0002)
oTrainer.nLogInterval  = 25
oTrainer.nSaveInterval = 100
oTrainer.cOutputDir    = "adapters/ring_lora_cloud"

# 4. Execute 3 Full Training Epochs
oTrainer.train(3)


## 📦 Step 8: Package & Download Trained LoRA Adapters
Compress all trained LoRA weights (`.tensor` and `.json`) into a zip archive and download directly to your local machine.

In [ ]:
from google.colab import files

# Compress adapters
!zip -r ring_lora_cloud.zip adapters/ring_lora_cloud/final/

# Trigger browser download
files.download("ring_lora_cloud.zip")
print("✅ LoRA adapter weights packaged and ready for local deployment!")